# Classification Metrics from Scratch — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

Evaluation code is where silent bugs live. A transposed confusion matrix, a "macro-F1" that is really a harmonic mean, an AUC that depends on how ties happen to be ordered, an "average precision" computed as a trapezoid, a calibration error that drops every 100%-confident prediction: each of these produces a plausible number that is wrong. Interviewers for applied-ML and research roles like these metrics because getting them exactly right means knowing what they measure.

Implement, in vectorized PyTorch, the metrics behind `sklearn.metrics.confusion_matrix`, `precision_recall_fscore_support`, `roc_auc_score`, `average_precision_score`, and the Expected Calibration Error of Guo et al., *On Calibration of Modern Neural Networks* (ICML 2017).

---

### Requirements

1. **`confusion_matrix(y_true, y_pred, num_classes) -> LongTensor`** of shape `(C, C)`: entry `[i, j]` counts samples whose **true** class is `i` and **predicted** class is `j` (rows = truth, columns = prediction). No Python loop over samples — a single counting op is enough.

2. **`precision_recall_f1(y_true, y_pred, num_classes, average) -> (precision, recall, f1)`** for single-label multiclass predictions, with **0/0 := 0** everywhere:
   - per class: $P_c = \frac{TP_c}{TP_c + FP_c}$, $R_c = \frac{TP_c}{TP_c + FN_c}$, $F1_c = \frac{2 P_c R_c}{P_c + R_c} = \frac{2\,TP_c}{2\,TP_c + FP_c + FN_c}$;
   - `average=None` → three `(C,)` float tensors (classes that never occur get 0);
   - `average="macro"` → the **unweighted mean of the per-class values** over the classes that occur in `y_true` or `y_pred` ($TP_c + FP_c + FN_c > 0$; what sklearn does by default and torchmetrics does for multiclass). Macro-F1 is the mean of per-class F1, **not** the harmonic mean of macro-P and macro-R;
   - `average="micro"` → pool $TP$, $FP$, $FN$ over classes, then apply the formulas once.

   Scalars may be returned as 0-dim tensors or Python floats.

3. **`roc_auc(scores, labels) -> float`** — binary. `labels` holds 0/1 (any integer or bool dtype) and `scores` are real numbers (higher = more positive). ROC-AUC is the probability that a random positive outranks a random negative, with **ties counting ½**:
   $$\mathrm{AUC} = \frac{1}{n_+ n_-}\sum_{i \in +}\sum_{j \in -}\Big(\mathbb{1}[s_i > s_j] + \tfrac12 \mathbb{1}[s_i = s_j]\Big)$$
   Compute it in **$O(n \log n)$** with ranks (the Mann–Whitney $U$ statistic): give tied scores their **average rank**, then $\mathrm{AUC} = \big(\sum_{i \in +} \mathrm{rank}_i - \tfrac{n_+(n_+ + 1)}{2}\big) / (n_+ n_-)$. Never build the $n_+ \times n_-$ comparison matrix. You may assume both classes are present.

4. **`average_precision(scores, labels) -> float`** — sklearn's step-wise definition. With the distinct score values sorted in decreasing order $t_1 > t_2 > \dots$, and $P_n, R_n$ the precision and recall of "predict positive iff score $\ge t_n$":
   $$\mathrm{AP} = \sum_n (R_n - R_{n-1})\, P_n, \qquad R_0 = 0.$$
   Tied scores form **one** threshold. There is no interpolation and no trapezoid. $O(n \log n)$. You may assume at least one positive.

5. **`expected_calibration_error(probs, labels, n_bins=15) -> float`** — `probs` is `(N, C)` with rows summing to 1, `labels` is `(N,)`. The confidence is $\hat p_i = \max_c \texttt{probs}[i, c]$ and the prediction is the argmax. Use `n_bins` **equal-width, right-closed** bins $B_b = \{i : \hat p_i \in (\tfrac{b}{n}, \tfrac{b+1}{n}]\}$, so a confidence of exactly 1.0 lands in the last bin:
   $$\mathrm{ECE} = \sum_{b} \frac{|B_b|}{N}\,\big|\mathrm{acc}(B_b) - \mathrm{conf}(B_b)\big|,$$
   where $\mathrm{acc}$ and $\mathrm{conf}$ are the mean correctness and mean confidence inside a bin. Empty bins contribute 0.

---

### Constraints

- ✅ Pure PyTorch, vectorized: `bincount`, `sort`, `cumsum`, `unique_consecutive`, `bucketize`, `scatter_add` are your friends.
- ❌ No `sklearn`, `torchmetrics` or `scipy` in your implementation (`sklearn` is used only in the optional cross-check cell).
- ❌ No Python loop over samples, and no $O(n^2)$ pairwise or all-thresholds matrices for AUC/AP.
- ✅ A loop over the `n_bins` bins in ECE is acceptable (the reference code of Guo et al. does that), though a `bincount` version is neater.

---

<details>
  <summary>💡 Hint</summary>

  - How many distinct `(true, pred)` pairs are there? Can you give each one a unique integer id?
  - Once you have the confusion matrix, where do TP, FP and FN live — the diagonal, the rows, the columns?
  - For AUC: if you sort all scores ascending, what does the rank of a positive sample tell you about how many negatives it beats? What should tied samples receive so that each positive–negative tie counts exactly ½?
  - For AP: after sorting by score (descending), a cumulative sum of the labels gives the TP count at every cut-off, but you may only cut *between* different score values.
  - ECE can be written as $\frac1N \sum_b \big|\sum_{i \in B_b} (\text{correct}_i - \hat p_i)\big|$.

</details>

---

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
torch.manual_seed(0)
num_classes, N = 4, 2000

# A synthetic 4-class classifier that is over-confident: informative logits sharpened by T = 0.5.
y_true = torch.randint(0, num_classes, (N,))
logits = torch.randn(N, num_classes) + 1.5 * F.one_hot(y_true, num_classes)
probs = torch.softmax(logits / 0.5, dim=1)
y_pred = probs.argmax(dim=1)

# A binary problem with ~10% positives and heavily tied (rounded) scores.
y_bin = (torch.rand(N) < 0.1).long()
s_bin = ((torch.randn(N) + 1.2 * y_bin) * 4).round() / 4
print(f"accuracy {(y_true == y_pred).float().mean():.3f}, positives {int(y_bin.sum())}/{N}, "
      f"distinct binary scores {s_bin.unique().numel()}")

In [ ]:
def confusion_matrix(y_true: torch.Tensor, y_pred: torch.Tensor, num_classes: int) -> torch.Tensor:
    """
    Args:
        y_true: (N,) int64 true classes in [0, num_classes).
        y_pred: (N,) int64 predicted classes in [0, num_classes).
        num_classes: C.

    Returns:
        (C, C) int64 tensor; [i, j] = #samples with true class i predicted as j.
    """
    pair_id = y_true.reshape(-1).long() * num_classes + y_pred.reshape(-1).long()   # in [0, C^2)
    counts = torch.bincount(pair_id, minlength=num_classes * num_classes)
    return counts.reshape(num_classes, num_classes)            # row = true, column = predicted

In [ ]:
def confusion_loop(y, p, C):
    cm = torch.zeros(C, C, dtype=torch.long)
    for t, q in zip(y.tolist(), p.tolist()):
        cm[t, q] += 1
    return cm

cm = confusion_matrix(y_true, y_pred, num_classes)
print(cm)
assert cm.dtype == torch.long and cm.shape == (num_classes, num_classes)
assert torch.equal(cm, confusion_loop(y_true, y_pred, num_classes))
assert confusion_matrix(torch.tensor([0, 0, 1]), torch.tensor([1, 1, 1]), 2).tolist() == [[0, 2], [0, 1]], \
    "rows must be the TRUE class"
print("confusion_matrix tests passed")

In [ ]:
def _safe_div(num: torch.Tensor, den: torch.Tensor) -> torch.Tensor:
    # Here num <= den and both are counts, so den == 0 implies num == 0: dividing by
    # max(den, 1) returns exactly 0 for 0/0 and the true ratio otherwise.
    return num / den.clamp(min=1)


def precision_recall_f1(y_true: torch.Tensor, y_pred: torch.Tensor, num_classes: int,
                        average="macro"):
    """
    Args:
        y_true, y_pred: (N,) int64 class ids.
        num_classes: C.
        average: None (per class), "macro" or "micro".

    Returns:
        (precision, recall, f1): (C,) tensors for average=None, scalars otherwise. 0/0 := 0.
    """
    if average not in (None, "macro", "micro"):
        raise ValueError(f"unknown average={average!r}")
    cm = confusion_matrix(y_true, y_pred, num_classes).double()
    tp = cm.diagonal()
    fp = cm.sum(dim=0) - tp        # column sum: predicted c, minus the correct ones
    fn = cm.sum(dim=1) - tp        # row sum: truly c, minus the correct ones
    if average == "micro":         # pool the counts over classes, then divide once
        tp, fp, fn = tp.sum(), fp.sum(), fn.sum()

    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    f1 = _safe_div(2 * tp, 2 * tp + fp + fn)       # = 2PR / (P + R), with 0/0 := 0

    if average == "macro":         # unweighted mean over classes that occur at all
        present = (tp + fp + fn) > 0
        k = present.sum().clamp(min=1)
        precision, recall, f1 = (x[present].sum() / k for x in (precision, recall, f1))
    return precision, recall, f1

In [ ]:
P, R, F1 = precision_recall_f1(y_true, y_pred, num_classes, None)
print("per-class P", P, "\nper-class R", R, "\nper-class F1", F1)
acc = (y_true == y_pred).double().mean()
mP, mR, mF = precision_recall_f1(y_true, y_pred, num_classes, "micro")
assert torch.allclose(torch.stack([torch.as_tensor(v).double() for v in (mP, mR, mF)]), acc.expand(3)), \
    "single-label multiclass: micro P = R = F1 = accuracy"
MP, MR, MF = precision_recall_f1(y_true, y_pred, num_classes, "macro")
assert abs(float(MF) - float(F1.mean())) < 1e-9, "macro-F1 is the mean of the per-class F1 scores"

# A class that is never predicted: precision 0/0 -> 0, no NaN
y = torch.tensor([0, 1, 2, 2]); p = torch.tensor([0, 1, 1, 1])
P, R, F1 = precision_recall_f1(y, p, 3, None)
assert not torch.isnan(P).any() and float(P[2]) == 0.0 and float(R[2]) == 0.0
print("precision_recall_f1 tests passed")

In [ ]:
def roc_auc(scores: torch.Tensor, labels: torch.Tensor) -> float:
    """
    Binary ROC-AUC = P(score_pos > score_neg) + 0.5 * P(tie), in O(n log n).

    Args:
        scores: (N,) real-valued scores.
        labels: (N,) 0/1 labels (int or bool).

    Returns:
        AUC as a Python float.
    """
    scores = scores.reshape(-1).double()        # float64: rank sums grow like n^2
    pos = labels.reshape(-1).bool()
    n_pos = pos.sum().double()
    n_neg = pos.numel() - n_pos

    sorted_scores, order = torch.sort(scores)                   # ascending
    _, counts = torch.unique_consecutive(sorted_scores, return_counts=True)
    last = counts.cumsum(0).double()             # 1-based rank of the last member of each tie run
    avg_rank = last - (counts.double() - 1) / 2  # mean of ranks (last - count + 1) ... last
    ranks = torch.empty_like(scores)
    ranks[order] = avg_rank.repeat_interleave(counts)           # back to input order

    # Mann-Whitney U: the rank of a positive counts every sample at or below it. Subtracting
    # n_pos (n_pos + 1) / 2 removes the positive-positive pairs; average ranks make each
    # positive-negative tie worth exactly 1/2.
    u = ranks[pos].sum() - n_pos * (n_pos + 1) / 2
    return float(u / (n_pos * n_neg))

In [ ]:
def auc_pairwise(s, y):
    # O(n^2) definition, fine for a test
    s, y = s.double(), y.bool()
    d = s[y][:, None] - s[~y][None, :]
    return float((d > 0).double().mean() + 0.5 * (d == 0).double().mean())

auc = roc_auc(s_bin, y_bin)
print(f"AUC = {auc:.6f}")
assert abs(auc - auc_pairwise(s_bin, y_bin)) < 1e-9, "ties must count 1/2"
assert abs(roc_auc(s_bin, y_bin.bool()) - auc) < 1e-12, "bool labels work too"
yy = torch.tensor([0, 0, 1, 1]); ss = torch.tensor([0.1, 0.4, 0.35, 0.8])
assert abs(roc_auc(ss, yy) - 0.75) < 1e-9
assert abs(roc_auc(-ss, yy) - 0.25) < 1e-9
assert abs(roc_auc(torch.ones(4), yy) - 0.5) < 1e-9, "constant scores -> 0.5"
print("roc_auc tests passed")

In [ ]:
def average_precision(scores: torch.Tensor, labels: torch.Tensor) -> float:
    """
    sklearn-style AP = sum_n (R_n - R_{n-1}) * P_n over distinct score thresholds.

    Args:
        scores: (N,) real-valued scores.
        labels: (N,) 0/1 labels (int or bool).

    Returns:
        AP as a Python float.
    """
    scores = scores.reshape(-1).double()
    y = labels.reshape(-1).double()
    sorted_scores, order = torch.sort(scores, descending=True)
    tp = y[order].cumsum(0)                       # TPs among the top-k, for every k

    # Thresholding at score >= t includes the whole tie run, so evaluate at the run ends.
    _, counts = torch.unique_consecutive(sorted_scores, return_counts=True)
    last = counts.cumsum(0) - 1
    tp = tp[last]
    n_predicted = (last + 1).double()

    precision = tp / n_predicted
    recall = tp / tp[-1]                          # tp[-1] = total number of positives
    recall_step = torch.diff(recall, prepend=recall.new_zeros(1))
    return float((recall_step * precision).sum())

In [ ]:
def ap_by_thresholding(s, y):
    # Brute force: threshold at every distinct score, highest first.
    s, y = s.double(), y.double()
    ap, r_prev = 0.0, 0.0
    for t in s.unique().flip(0):
        sel = s >= t
        tp = float(y[sel].sum())
        ap += (tp / float(y.sum()) - r_prev) * tp / float(sel.sum())
        r_prev = tp / float(y.sum())
    return ap

ap = average_precision(s_bin, y_bin)
print(f"AP = {ap:.6f}   (positive rate {y_bin.double().mean():.3f})")
assert abs(ap - ap_by_thresholding(s_bin, y_bin)) < 1e-9
assert abs(average_precision(torch.tensor([.9, .8, .7, .6]), torch.tensor([1, 0, 1, 0])) - (0.5 + 0.5 * 2 / 3)) < 1e-9
assert abs(average_precision(torch.full((10,), 0.3), torch.tensor([1, 0, 0, 1, 0, 0, 0, 0, 0, 0])) - 0.2) < 1e-9, \
    "all scores tied -> one threshold -> AP = positive rate"
print("average_precision tests passed")

In [ ]:
def expected_calibration_error(probs: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> float:
    """
    Top-label ECE with equal-width, right-closed confidence bins (b/n, (b+1)/n].

    Args:
        probs: (N, C) predicted class probabilities (rows sum to 1).
        labels: (N,) int64 true classes.
        n_bins: number of bins.

    Returns:
        ECE as a Python float.
    """
    conf, pred = probs.max(dim=1)
    correct = (pred == labels).to(conf.dtype)
    edges = torch.linspace(0, 1, n_bins + 1, dtype=conf.dtype, device=conf.device)
    # bucketize(right=False) returns b with edges[b] < conf <= edges[b + 1] when given the
    # interior edges: right-closed bins, and conf == 1.0 falls in the last bin.
    bin_idx = torch.bucketize(conf, edges[1:-1])
    # |B|/N * |acc(B) - conf(B)| = |sum_{i in B} (correct_i - conf_i)| / N
    gap = torch.bincount(bin_idx, weights=correct - conf, minlength=n_bins)
    return float(gap.abs().sum() / conf.numel())

In [ ]:
def ece_loop(probs, labels, n_bins=15):
    conf, pred = probs.max(dim=1)
    correct = (pred == labels).double()
    ece = 0.0
    for b in range(n_bins):
        in_bin = (conf > b / n_bins) & (conf <= (b + 1) / n_bins)
        if in_bin.any():
            ece += in_bin.double().mean() * (correct[in_bin].mean() - conf[in_bin].double().mean()).abs()
    return float(ece)

ece = expected_calibration_error(probs, y_true)
print(f"ECE (T = 0.5, over-confident) = {ece:.4f}")
assert abs(ece - ece_loop(probs, y_true)) < 1e-6
# 75%-confident and right 3 times out of 4, plus certain and right: perfectly calibrated
calibrated = torch.tensor([[0.75, 0.25]] * 4 + [[0.0, 1.0]] * 2)
assert abs(expected_calibration_error(calibrated, torch.tensor([0, 0, 0, 1, 1, 1]))) < 1e-9
# confidence exactly 1.0 must be counted (last bin is right-closed)
assert abs(expected_calibration_error(torch.tensor([[1.0, 0.0]]), torch.tensor([1])) - 1.0) < 1e-9
print("expected_calibration_error tests passed")

In [ ]:
# Optional: cross-check against scikit-learn if it is installed (it is on Colab).
try:
    from sklearn import metrics as skm
except ImportError:
    print("scikit-learn not installed -- skipping the cross-check")
else:
    assert (confusion_matrix(y_true, y_pred, num_classes).numpy()
            == skm.confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))).all()
    for avg in ("macro", "micro"):
        ours = [float(v) for v in precision_recall_f1(y_true, y_pred, num_classes, avg)]
        theirs = skm.precision_recall_fscore_support(y_true, y_pred, average=avg, zero_division=0)[:3]
        assert max(abs(a - b) for a, b in zip(ours, theirs)) < 1e-9, (avg, ours, theirs)
    assert abs(roc_auc(s_bin, y_bin) - skm.roc_auc_score(y_bin, s_bin)) < 1e-9
    assert abs(average_precision(s_bin, y_bin) - skm.average_precision_score(y_bin, s_bin)) < 1e-9
    print("matches sklearn.metrics")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("classification-metrics", confusion_matrix=confusion_matrix,
      precision_recall_f1=precision_recall_f1, roc_auc=roc_auc,
      average_precision=average_precision, expected_calibration_error=expected_calibration_error)

In [ ]:
# Optional demo: temperature scaling (Guo et al., 2017). Fit one scalar T on held-out data by
# minimising NLL; argmax (and so accuracy) is unchanged, while calibration improves.
torch.manual_seed(1)
yv = torch.randint(0, num_classes, (N,))
zv = (torch.randn(N, num_classes) + 1.5 * F.one_hot(yv, num_classes)) / 0.5   # same over-confident model
temps = torch.linspace(0.5, 5.0, 91)
nll = torch.stack([F.cross_entropy(zv / t, yv) for t in temps])
T = float(temps[nll.argmin()])
z_test = logits / 0.5
print(f"fitted T = {T:.2f}")
print(f"ECE before: {expected_calibration_error(torch.softmax(z_test, 1), y_true):.4f}   "
      f"after: {expected_calibration_error(torch.softmax(z_test / T, 1), y_true):.4f}")

### Why it works

- **Confusion matrix.** $(i, j) \mapsto i \cdot C + j$ is a bijection between label pairs and $[0, C^2)$, so one `bincount` counts every cell, and a reshape puts true classes on the rows. From it: $TP$ = diagonal, $FP$ = column sum − diagonal, $FN$ = row sum − diagonal.
- **Micro = accuracy.** In single-label multiclass, every error is exactly one FP (for the predicted class) and one FN (for the true class), so $\sum FP = \sum FN = N - \sum TP$. Pooled precision and pooled recall both equal $\sum TP / N$.
- **AUC = normalised Mann–Whitney $U$.** With ranks $1..n$ in ascending score order, a positive at rank $r$ has $r - 1$ samples below it. Summing over positives and subtracting the $n_+(n_+ - 1)/2$ positive–positive pairs (together with the $n_+$ self-counts, that is $n_+(n_+ + 1)/2$) counts the negatives each positive beats. Giving a tie run its average rank splits every tied positive–negative pair ½–½. That matches the trapezoidal area under the ROC curve exactly.
- **AP.** Precision and recall only change when the threshold crosses a score value, and "score ≥ t" includes the whole tie run, so they are evaluated at the end of each run. The sum $\sum (R_n - R_{n-1}) P_n$ is the area under the PR *step* function. That avoids the optimistic linear interpolation of the trapezoid: interpolating linearly in PR space is not achievable by any classifier (Davis & Goadrich, 2006).
- **ECE identity.** $\frac{|B|}{N}\,|\mathrm{acc}(B) - \mathrm{conf}(B)| = \frac{1}{N}\,\big|\sum_{i \in B} (\text{correct}_i - \hat p_i)\big|$, so a single weighted `bincount` gives every bin's signed gap.

### Complexity

- `confusion_matrix`: $O(N + C^2)$.
- `precision_recall_f1`: $O(N + C^2)$.
- `roc_auc` and `average_precision`: $O(n \log n)$ for the sort, the rest is linear. Memory is $O(n)$, versus $O(n_+ n_-)$ for the pairwise definition.
- `expected_calibration_error`: $O(NC + n_{\text{bins}})$.

### Common bugs

- **Transposed confusion matrix**: sklearn uses rows = truth. Or a Python loop over samples.
- **Macro-F1 as $2\bar P \bar R / (\bar P + \bar R)$**, the harmonic mean of macro-P and macro-R. That is a different and less common number.
- **NaN for empty classes**: precision of a never-predicted class is 0/0. Also ambiguity about which classes enter the macro mean; state the convention.
- **AUC with ordinal ranks** (`argsort().argsort()`): ties get distinct ranks, so the AUC depends on the input order of tied samples. Also strict `>` in the pairwise formula without the ½ for ties, and float32 rank sums at large $n$.
- **AP as a trapezoid** (`auc(recall, precision)`), with interpolation, or with one step per sample instead of per distinct score (order-dependent under ties).
- **ECE with left-closed bins** `[lo, hi)` (e.g. `torch.bucketize(..., right=True)`): a confidence of exactly 1.0 falls outside every bin and is silently dropped unless clamped, and confidences on an interior edge move up a bin. Also an unweighted average over bins, or using the probability of the *true* label instead of the max probability.

### Interview follow-ups

1. **When do you use ROC-AUC vs PR-AUC (AP) under class imbalance?** TPR and FPR are each normalised within one class, so ROC-AUC does not change with prevalence. At 1:1000 a model can reach AUC 0.99 and still have poor precision, because the many negatives make a small FPR a large number of false alarms. AP's baseline equals the positive rate and it focuses on the positives, so it is the better summary for rare-event detection and retrieval. ROC-AUC is the right choice when you care about ranking quality across both classes, or when prevalence will shift at deployment.
2. **Micro vs macro vs weighted?** Micro pools counts, so frequent classes dominate (it equals accuracy for single-label problems). Macro weights every class equally, which exposes poor rare-class performance. Weighted averages by support, and its F1 need not lie between its P and R. For multilabel problems micro ≠ accuracy, and "samples" averaging is another option.
3. **What is calibration, and how do you fix it?** A model is calibrated when, among predictions made with confidence $p$, a fraction $p$ is correct. Modern deep nets tend to be over-confident (Guo et al.). **Temperature scaling** fits one scalar $T$ on a held-out set by minimising NLL of $\mathrm{softmax}(z/T)$. It cannot change the argmax, so accuracy is preserved, and $T > 1$ softens over-confident logits. Alternatives: Platt scaling (binary), isotonic regression, histogram binning, vector/matrix/Dirichlet scaling. Training-time fixes such as label smoothing, mixup and focal loss also affect calibration.
4. **What are ECE's weaknesses?** It depends on the binning (equal-width vs equal-mass/adaptive ECE) and on `n_bins`, and it is a biased estimator. It only checks top-label confidence (classwise ECE extends it). A useless model that always predicts the class prior with confidence equal to its accuracy has ECE 0. So report it next to a proper scoring rule (NLL or the Brier score), plus a reliability diagram.
5. **How do you compute AUC on a stream or across workers?** Exact AUC needs a global sort. Approximate it by histogramming scores into fixed bins per class (counts are mergeable across workers), then integrating the binned ROC with cumulative sums. `tf.keras.metrics.AUC(num_thresholds=200)` works this way. The error shrinks with finer bins, and ties inside a bin count ½.
6. **How do you choose a decision threshold?** AUC and AP are threshold-free. Deployment needs an operating point: maximise F1 or F$_\beta$, fix precision at a required recall (or the reverse), or minimise expected cost from the confusion matrix. Pick the threshold on validation data, and recheck calibration if priors shift.